# Architecture 04: Document Processing Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/04-document-processing.ipynb)

**Companion notebook for**: `04-document-processing.html`

## Learning Objectives
- Extract text from PDF documents programmatically
- Use LLMs with JSON mode for structured data extraction
- Build Pydantic models for schema-driven extraction
- Create a batch document processing pipeline
- Validate and post-process extracted outputs

## Prerequisites
- Python 3.10+
- OpenAI API key (optional — mock fallbacks included)

---
## Setup & Dependencies

In [ ]:
%pip install -q openai pydantic PyPDF2

In [ ]:
import os
import json
import re
from typing import Optional

from openai import OpenAI

API_KEY = os.environ.get("OPENAI_API_KEY", "")
client = OpenAI(api_key=API_KEY) if API_KEY else None

if not API_KEY:
    print("⚠️  OPENAI_API_KEY not set. Mock responses will be used.")
else:
    print("✅ API key loaded.")

---
## Section 1: Simulating PDF Text Extraction

In production you'd use PyPDF2, pdfplumber, or cloud OCR services.
Here we simulate extracted text from different document types.

In [ ]:
# Simulated extracted text from different document types

SAMPLE_INVOICE = """INVOICE
Invoice Number: INV-2024-0847
Date: March 15, 2024
Due Date: April 14, 2024

Bill To:
Acme Corporation
123 Business Ave, Suite 400
San Francisco, CA 94105

From:
TechSupply Inc.
456 Vendor Lane
Austin, TX 78701

Items:
1. Cloud Hosting (Annual)    $12,000.00
2. SSL Certificate (3-year)  $299.00
3. Support Plan (Premium)    $4,800.00

Subtotal: $17,099.00
Tax (8.25%): $1,410.67
Total: $18,509.67

Payment Terms: Net 30
Bank: First National Bank
Account: 9876543210
Routing: 021000021"""

SAMPLE_CONTRACT = """SERVICE AGREEMENT

This Service Agreement ("Agreement") is entered into as of January 1, 2024,
by and between DataFlow Systems LLC ("Provider") and GlobalRetail Inc. ("Client").

1. SERVICES: Provider shall deliver cloud-based data analytics services
   including real-time dashboard, ETL pipeline management, and monthly reporting.

2. TERM: This Agreement shall commence on January 1, 2024 and continue
   for a period of twenty-four (24) months, ending December 31, 2025.

3. COMPENSATION: Client shall pay Provider a monthly fee of $15,000 USD,
   due on the first business day of each month. Late payments incur 1.5%
   monthly interest.

4. TERMINATION: Either party may terminate with 90 days written notice.
   Early termination fee: 3 months of service fees.

5. CONFIDENTIALITY: Both parties agree to maintain confidentiality of
   proprietary information for 3 years after termination.

6. LIABILITY: Provider's total liability shall not exceed 12 months of fees paid.

Signed:
DataFlow Systems LLC - Jane Smith, CEO
GlobalRetail Inc. - Robert Chen, CTO"""

SAMPLE_RESUME = """SARAH JOHNSON
Senior Machine Learning Engineer
sarah.johnson@email.com | (555) 123-4567 | San Francisco, CA
LinkedIn: linkedin.com/in/sarahjohnson | GitHub: github.com/sjohnson

EXPERIENCE

Senior ML Engineer — TechCorp AI (2022-Present)
- Led development of real-time recommendation system serving 10M+ users
- Reduced model inference latency by 40% through TensorRT optimization
- Built MLOps pipeline with Kubeflow, reducing deployment time from days to hours

ML Engineer — DataStart Inc. (2019-2022)
- Developed NLP models for sentiment analysis achieving 94% accuracy
- Implemented A/B testing framework for model comparison
- Mentored 3 junior engineers

EDUCATION
M.S. Computer Science — Stanford University (2019)
B.S. Mathematics — UC Berkeley (2017)

SKILLS
Python, PyTorch, TensorFlow, Kubernetes, AWS, GCP, SQL, Spark"""

documents = {
    "invoice": SAMPLE_INVOICE,
    "contract": SAMPLE_CONTRACT,
    "resume": SAMPLE_RESUME,
}

print(f"Loaded {len(documents)} sample documents.")
for name, text in documents.items():
    print(f"  - {name}: {len(text)} chars, {len(text.splitlines())} lines")

---
## Section 2: Basic LLM Extraction (Unstructured)

The simplest approach: send the document text to an LLM and ask it to extract key fields.
This works but gives inconsistent output formats.

In [ ]:
def extract_basic(document_text: str, document_type: str) -> str:
    """Basic extraction — ask the LLM to pull out key information."""
    prompt = f"""Extract the key information from this {document_type}.
List each field on its own line as 'Field: Value'.

Document:
---
{document_text}
---"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=500,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"API call failed: {e}")
        # Mock response
        if document_type == "invoice":
            return (
                "Invoice Number: INV-2024-0847\n"
                "Date: March 15, 2024\n"
                "Due Date: April 14, 2024\n"
                "Vendor: TechSupply Inc.\n"
                "Client: Acme Corporation\n"
                "Subtotal: $17,099.00\n"
                "Tax: $1,410.67\n"
                "Total: $18,509.67\n"
                "Payment Terms: Net 30"
            )
        return "Mock: Extracted fields for " + document_type


result = extract_basic(SAMPLE_INVOICE, "invoice")
print("Basic extraction result:")
print(result)
print(f"\n⚠️  Problem: output is plain text — hard to parse programmatically.")

---
## Section 3: Structured Extraction with JSON Mode

Use OpenAI's JSON mode (`response_format={"type": "json_object"}`) to get
guaranteed valid JSON output. This is much more reliable for downstream processing.

In [ ]:
def extract_json(document_text: str, document_type: str, fields: list[str]) -> dict:
    """Extract specific fields as JSON using JSON mode."""
    fields_str = ", ".join(f'"{f}"' for f in fields)
    prompt = f"""Extract the following fields from this {document_type}: {fields_str}.

Return a JSON object with these exact keys. Use null for any field not found.
For monetary values, return as numbers (no $ sign). For dates, use YYYY-MM-DD format.

Document:
---
{document_text}
---"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You extract structured data from documents. Always respond with valid JSON."},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
            max_tokens=500,
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"API call failed: {e}. Using mock.")
        return {
            "invoice_number": "INV-2024-0847",
            "date": "2024-03-15",
            "due_date": "2024-04-14",
            "vendor_name": "TechSupply Inc.",
            "client_name": "Acme Corporation",
            "subtotal": 17099.00,
            "tax": 1410.67,
            "total": 18509.67,
            "payment_terms": "Net 30",
        }


invoice_fields = [
    "invoice_number", "date", "due_date", "vendor_name",
    "client_name", "subtotal", "tax", "total", "payment_terms",
]

result = extract_json(SAMPLE_INVOICE, "invoice", invoice_fields)
print("Structured JSON extraction:")
print(json.dumps(result, indent=2))
print(f"\n✅ Output is a Python dict — ready for database insertion or API response.")

---
## Section 4: Schema-Driven Extraction with Pydantic

For production systems, define Pydantic models for each document type.
This gives you type validation, default values, and automatic documentation.

In [ ]:
from pydantic import BaseModel, Field, field_validator
from datetime import date


class LineItem(BaseModel):
    description: str
    amount: float


class InvoiceExtraction(BaseModel):
    invoice_number: str
    date: date
    due_date: date
    vendor_name: str
    client_name: str
    line_items: list[LineItem]
    subtotal: float
    tax: float
    total: float
    payment_terms: str = "Net 30"

    @field_validator("total")
    @classmethod
    def total_must_match(cls, v, info):
        expected = info.data.get("subtotal", 0) + info.data.get("tax", 0)
        if abs(v - expected) > 0.01:
            raise ValueError(f"Total {v} != subtotal + tax ({expected})")
        return v


class ContractExtraction(BaseModel):
    provider: str
    client: str
    start_date: date
    end_date: date
    term_months: int
    monthly_fee: float
    termination_notice_days: int
    early_termination_fee_months: int
    liability_cap_months: int
    confidentiality_years: int
    services: list[str]


class ResumeExtraction(BaseModel):
    name: str
    title: str
    email: str
    phone: Optional[str] = None
    location: Optional[str] = None
    years_experience: int
    skills: list[str]
    education: list[str]
    current_company: Optional[str] = None


# Show the JSON schema for the invoice model
print("InvoiceExtraction JSON Schema:")
print(json.dumps(InvoiceExtraction.model_json_schema(), indent=2))

In [ ]:
def extract_with_schema(document_text: str, schema_model: type[BaseModel]) -> BaseModel:
    """Extract data using a Pydantic schema to guide the LLM."""
    schema = json.dumps(schema_model.model_json_schema(), indent=2)
    prompt = f"""Extract data from the following document.
Return a JSON object matching this exact schema:

{schema}

Document:
---
{document_text}
---"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a document extraction engine. Return only valid JSON matching the provided schema."},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
            max_tokens=800,
            response_format={"type": "json_object"},
        )
        data = json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"API call failed: {e}. Using mock data.")
        if schema_model == ContractExtraction:
            data = {
                "provider": "DataFlow Systems LLC",
                "client": "GlobalRetail Inc.",
                "start_date": "2024-01-01",
                "end_date": "2025-12-31",
                "term_months": 24,
                "monthly_fee": 15000.0,
                "termination_notice_days": 90,
                "early_termination_fee_months": 3,
                "liability_cap_months": 12,
                "confidentiality_years": 3,
                "services": ["real-time dashboard", "ETL pipeline management", "monthly reporting"],
            }
        elif schema_model == ResumeExtraction:
            data = {
                "name": "Sarah Johnson",
                "title": "Senior Machine Learning Engineer",
                "email": "sarah.johnson@email.com",
                "phone": "(555) 123-4567",
                "location": "San Francisco, CA",
                "years_experience": 5,
                "skills": ["Python", "PyTorch", "TensorFlow", "Kubernetes", "AWS", "GCP", "SQL", "Spark"],
                "education": ["M.S. Computer Science — Stanford University (2019)", "B.S. Mathematics — UC Berkeley (2017)"],
                "current_company": "TechCorp AI",
            }
        else:
            data = {}

    # Validate with Pydantic
    return schema_model.model_validate(data)


# Extract contract
contract = extract_with_schema(SAMPLE_CONTRACT, ContractExtraction)
print("Contract Extraction:")
print(json.dumps(contract.model_dump(mode="json"), indent=2))
print(f"\nTotal contract value: ${contract.monthly_fee * contract.term_months:,.2f}")
print(f"Early exit cost: ${contract.monthly_fee * contract.early_termination_fee_months:,.2f}")

In [ ]:
# Extract resume
resume = extract_with_schema(SAMPLE_RESUME, ResumeExtraction)
print("Resume Extraction:")
print(json.dumps(resume.model_dump(mode="json"), indent=2))
print(f"\nCandidate: {resume.name}")
print(f"Skills match: {', '.join(resume.skills[:5])}...")
print(f"Experience: {resume.years_experience} years")

---
## Section 5: Batch Document Processing Pipeline

Process multiple documents through the extraction pipeline with logging,
error handling, and result aggregation.

In [ ]:
import time
from dataclasses import dataclass, field


@dataclass
class ProcessingResult:
    doc_id: str
    doc_type: str
    status: str  # "success" | "error" | "validation_failed"
    data: dict = field(default_factory=dict)
    error: str = ""
    processing_time_ms: float = 0.0


# Schema registry — maps document types to Pydantic models
SCHEMA_REGISTRY = {
    "contract": ContractExtraction,
    "resume": ResumeExtraction,
    # InvoiceExtraction has a validator that checks total — skip for batch demo
}


def process_document(doc_id: str, doc_type: str, text: str) -> ProcessingResult:
    """Process a single document through the extraction pipeline."""
    start = time.time()

    if doc_type not in SCHEMA_REGISTRY:
        return ProcessingResult(
            doc_id=doc_id, doc_type=doc_type,
            status="error", error=f"Unknown doc type: {doc_type}",
        )

    try:
        schema = SCHEMA_REGISTRY[doc_type]
        result = extract_with_schema(text, schema)
        elapsed = (time.time() - start) * 1000
        return ProcessingResult(
            doc_id=doc_id, doc_type=doc_type,
            status="success",
            data=result.model_dump(mode="json"),
            processing_time_ms=elapsed,
        )
    except Exception as e:
        elapsed = (time.time() - start) * 1000
        return ProcessingResult(
            doc_id=doc_id, doc_type=doc_type,
            status="error", error=str(e),
            processing_time_ms=elapsed,
        )


# --- Batch processing ---
batch = [
    ("DOC-001", "contract", SAMPLE_CONTRACT),
    ("DOC-002", "resume", SAMPLE_RESUME),
    ("DOC-003", "contract", SAMPLE_CONTRACT),  # duplicate for demo
    ("DOC-004", "resume", SAMPLE_RESUME),
]

results = []
print(f"Processing {len(batch)} documents...\n")
print(f"{'ID':<10} {'Type':<12} {'Status':<18} {'Time (ms)':>10}")
print("-" * 52)

for doc_id, doc_type, text in batch:
    r = process_document(doc_id, doc_type, text)
    results.append(r)
    print(f"{r.doc_id:<10} {r.doc_type:<12} {r.status:<18} {r.processing_time_ms:>8.1f}ms")

# Summary
success = sum(1 for r in results if r.status == "success")
failed = len(results) - success
avg_time = sum(r.processing_time_ms for r in results) / len(results)
print(f"\n--- Summary ---")
print(f"Total: {len(results)} | Success: {success} | Failed: {failed}")
print(f"Avg processing time: {avg_time:.1f}ms")

---
## Section 6: Output Validation & Post-Processing

After extraction, validate the output for completeness, consistency,
and business rules before storing.

In [ ]:
def validate_contract(data: dict) -> list[str]:
    """Run business-rule validations on an extracted contract."""
    issues = []

    # Check date consistency
    if data.get("start_date") and data.get("end_date"):
        if data["start_date"] >= data["end_date"]:
            issues.append("Start date must be before end date.")

    # Check term matches date range
    if data.get("term_months") and data.get("start_date") and data.get("end_date"):
        from dateutil.relativedelta import relativedelta
        try:
            start = date.fromisoformat(str(data["start_date"]))
            end = date.fromisoformat(str(data["end_date"]))
            expected = (end.year - start.year) * 12 + (end.month - start.month)
            if abs(data["term_months"] - expected) > 1:
                issues.append(f"Term ({data['term_months']}mo) doesn't match date range ({expected}mo).")
        except Exception:
            pass

    # Check fee is reasonable
    if data.get("monthly_fee", 0) <= 0:
        issues.append("Monthly fee must be positive.")

    # Check required fields
    for field_name in ["provider", "client", "services"]:
        if not data.get(field_name):
            issues.append(f"Missing required field: {field_name}")

    return issues


def validate_resume(data: dict) -> list[str]:
    """Run validations on an extracted resume."""
    issues = []

    # Check email format
    if data.get("email") and not re.match(r"[^@]+@[^@]+\.[^@]+", data["email"]):
        issues.append(f"Invalid email format: {data['email']}")

    # Check skills not empty
    if not data.get("skills"):
        issues.append("No skills extracted.")

    # Check years_experience is reasonable
    if data.get("years_experience", 0) > 50:
        issues.append(f"Unreasonable years of experience: {data['years_experience']}")

    return issues


VALIDATORS = {
    "contract": validate_contract,
    "resume": validate_resume,
}

# Validate all successful results
print("Post-Extraction Validation")
print("=" * 50)

for r in results:
    if r.status != "success":
        continue

    validator = VALIDATORS.get(r.doc_type)
    if validator:
        issues = validator(r.data)
        status = "PASS" if not issues else "WARN"
        print(f"\n{r.doc_id} ({r.doc_type}): {status}")
        if issues:
            for issue in issues:
                print(f"  ⚠️  {issue}")
        else:
            print(f"  ✅ All validations passed.")

---
## Section 7: Confidence Scoring & Human Review

For critical documents, add a confidence scoring step. Low-confidence
extractions get routed to human reviewers.

In [ ]:
def score_extraction_confidence(document_text: str, extracted: dict) -> dict:
    """Ask the LLM to score confidence on each extracted field."""
    prompt = f"""You are a document extraction auditor. Compare the extracted data against the original document.
For each field, assign a confidence score (0.0 to 1.0) and flag any field that seems incorrect.

Return JSON with format: {{"field_name": {{"confidence": 0.95, "issue": null}}, ...}}

Extracted data:
{json.dumps(extracted, indent=2, default=str)}

Original document:
---
{document_text[:1500]}
---"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=500,
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        # Mock confidence scores
        return {
            field: {"confidence": 0.95 if i < 5 else 0.85, "issue": None}
            for i, field in enumerate(extracted.keys())
        }


# Score the contract extraction
scores = score_extraction_confidence(SAMPLE_CONTRACT, contract.model_dump(mode="json"))

print("Extraction Confidence Scores")
print("=" * 50)
print(f"{'Field':<30} {'Confidence':>10} {'Status':>10}")
print("-" * 50)

needs_review = []
for field_name, info in scores.items():
    conf = info.get("confidence", 0)
    status = "✅" if conf >= 0.9 else "⚠️" if conf >= 0.7 else "❌"
    print(f"{field_name:<30} {conf:>8.0%} {status:>10}")
    if conf < 0.9:
        needs_review.append(field_name)

if needs_review:
    print(f"\n🔍 Fields requiring human review: {', '.join(needs_review)}")
else:
    print(f"\n✅ All fields above confidence threshold — auto-approved.")

---
## Summary

In this notebook we built a complete document processing pipeline:

1. **Text Extraction** — Simulated PDF/image text extraction (use PyPDF2, pdfplumber, or cloud OCR in production)

2. **Basic Extraction** — Simple LLM prompt to extract key fields (unstructured output)

3. **JSON Mode** — Use `response_format={"type": "json_object"}` for guaranteed valid JSON

4. **Schema-Driven** — Pydantic models define expected fields, types, and validation rules; the JSON schema guides the LLM

5. **Batch Processing** — Pipeline with error handling, timing, and result aggregation

6. **Validation** — Business-rule checks after extraction (date consistency, required fields, format checks)

7. **Confidence Scoring** — LLM-based audit with human-in-the-loop for low-confidence fields

**Key takeaway**: Always validate LLM extractions with Pydantic schemas and business rules. The LLM does the heavy lifting, but guardrails ensure production reliability.

**Next notebook**: Architecture 05 — Multi-Model Router